# Overview Materi

Source: https://www.youtube.com/watch?v=LDRbO9a6XPU

Jelaskan secara singkat apa itu decision tree menurut pemahamanmu!

Decision tree adalah salah satu algoritma dalam machine learning yang digunakan untuk klasifikasi atau regresi. Modelnya prediksi berbasis pertanyaan berulang yang bercabang hingga menemukan keputusan akhir.

# Import Data & Libraries

In [ ]:
from __future__ import print_function

# label kolom
header = ["color", "diameter", "label"]

# data training
training_data = [
    ['Green', 3, 'Apple'],
    ['Yellow', 3, 'Apple'],
    ['Red', 1, 'Grape'],
    ['Red', 1, 'Grape'],
    ['Yellow', 3, 'Lemon'],]

# data testing
testing_data = [
    ['Green', 3, 'Apple'],
    ['Yellow', 4, 'Apple'],
    ['Red', 2, 'Grape'],
    ['Red', 1, 'Grape'],
    ['Yellow', 3, 'Lemon'],]

# Fungsi Dasar

In [16]:
# fungsi mencari apa saja unique value dari suatu kolom
def unique_vals(rows, col):
  return set([row[col] for row in rows])

# contoh penggunaan
training_data = [
    ['Green', 3, 'Apple'],
    ['Yellow', 3, 'Apple'],
    ['Red', 1, 'Grape'],
    ['Red', 1, 'Grape'],
    ['Yellow', 3, 'Lemon'],]
print(unique_vals(training_data, 0))
print(unique_vals(training_data, 1))
print(unique_vals(training_data, 2))

{'Yellow', 'Green', 'Red'}
{1, 3}
{'Grape', 'Apple', 'Lemon'}


In [ ]:
# fungsi Menghitung jumlah unique value dari suatu kolom
def class_counts(rows):
    counts = {}
    for row in rows:
        label = row[-1]
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    return counts

# contoh penggunaan
training_data = [
    ['Green', 3, 'Apple'],
    ['Yellow', 3, 'Apple'],
    ['Red', 1, 'Grape'],
    ['Red', 1, 'Grape'],
    ['Yellow', 3, 'Lemon'],]
counts = class_counts(training_data)
print("Jumlah masing-masing buah di data pelatihan:")
print(counts)

Jumlah masing-masing buah di data pelatihan:
{'Apple': 2, 'Grape': 2, 'Lemon': 1}


In [ ]:
# fungsi pengecekan suatu value numerik atau bukan
def is_numeric(value):
    return isinstance(value, int) or isinstance(value, float)

# contoh penggunaan
print(is_numeric(7))
print(is_numeric("Red"))

True
False


In [27]:
# label kolom
header = ["color", "diameter", "label"]

# fungsi pengecekan suatu value numerik atau bukan
def is_numeric(value):
    return isinstance(value, int) or isinstance(value, float)

# kelas untuk merepresentasikan pertanyaan pada decision tree
class Question:

    # inisialisasi kolom dan nilai pertanyaan
    def __init__(self, column, value):
        self.column = column
        self.value = value

    # mengecek apakah contoh data sesuai dengan pertanyaan
    def match(self, example):
        val = example[self.column]
        if is_numeric(val):
            return val >= self.value
        else:
            return val == self.value

    # menampilkan pertanyaan dalam format string yang mudah dibaca
    def __repr__(self):
        condition = "=="
        if is_numeric(self.value):
            condition = ">="
        return "Is %s %s %s?" % (
            header[self.column], condition, str(self.value))

# contoh penggunaan 1
training_data = [
    ['Green', 3, 'Apple'],
    ['Yellow', 3, 'Apple'],
    ['Red', 1, 'Grape'],
    ['Red', 1, 'Grape'],
    ['Yellow', 3, 'Lemon'],]
q1 = Question(0, 'Red')
print(q1)
for row in training_data:
    match = q1.match(row)
    print(f"Apakah data {row} cocok? {match}")
    print("\n")
q2 = Question(1,3)
print(q2)
for row in training_data:
    match = q2.match(row)
    print(f"Apakah data {row} cocok? {match}")

Is color == Red?
Apakah data ['Green', 3, 'Apple'] cocok? False


Apakah data ['Yellow', 3, 'Apple'] cocok? False


Apakah data ['Red', 1, 'Grape'] cocok? True


Apakah data ['Red', 1, 'Grape'] cocok? True


Apakah data ['Yellow', 3, 'Lemon'] cocok? False


Is diameter >= 3?
Apakah data ['Green', 3, 'Apple'] cocok? True
Apakah data ['Yellow', 3, 'Apple'] cocok? True
Apakah data ['Red', 1, 'Grape'] cocok? False
Apakah data ['Red', 1, 'Grape'] cocok? False
Apakah data ['Yellow', 3, 'Lemon'] cocok? True


In [ ]:
# membagi dataset menjadi dua berdasarkan pertanyaan
def partition(rows, question):
    true_rows, false_rows = [], []
    for row in rows:
        if question.match(row):
            true_rows.append(row)
        else:
            false_rows.append(row)
    return true_rows, false_rows

# contoh penggunaan
q = Question(0, 'Red')
true_rows, false_rows = partition(training_data, q)
print("Data yang sesuai dengan pertanyaan:")
print(true_rows)
print("Data yang tidak sesuai dengan pertanyaan:")
print(false_rows)

Data yang sesuai dengan pertanyaan:
[['Red', 1, 'Grape'], ['Red', 1, 'Grape']]
Data yang tidak sesuai dengan pertanyaan:
[['Green', 3, 'Apple'], ['Yellow', 3, 'Apple'], ['Yellow', 3, 'Lemon']]


**apa itu gini impurity?**
<br> gini impurity berfungsi mengukur tingkat ketidakmurnian atau ketidakteraturan pada sebuah simpul (node) dalam pohon

In [ ]:
# menghitung nilai Gini Impurity untuk sebuah dataset
def gini(rows):
    counts = class_counts(rows)
    impurity = 1
    for lbl in counts:
        prob_of_lbl = counts[lbl] / float(len(rows))
        impurity -= prob_of_lbl**2
    return impurity

# contoh penggunaan
no_mixing = [['Apple'],
              ['Apple']]
mixing = [['Apple'],
          ['Orange']]
print(gini(no_mixing))
print(gini(mixing))

0.0
0.5


**apa itu information gain?**
<br> information gain berfungsi mengukur seberapa efektif sebuah fitur dalam memisahkan data berdasarkan kelas-kelasnya

In [ ]:
# menghitung nilai Information Gain dari pemisahan dataset
def info_gain(left, right, current_uncertainty):
    p = float(len(left)) / (len(left) + len(right))
    return current_uncertainty - p * gini(left) - (1 - p) * gini(right)

# contoh penggunaan
current_uncertainty = gini(training_data)
true_rows, false_rows = partition(training_data, Question(0, 'Green'))
info_gain(true_rows, false_rows, current_uncertainty)
print(info_gain(true_rows, false_rows, current_uncertainty))

0.1399999999999999


In [32]:
# label kolom
header = ["color", "diameter", "label"]

# fungsi pengecekan suatu value numerik atau bukan
def is_numeric(value):
    return isinstance(value, int) or isinstance(value, float)

# kelas untuk merepresentasikan pertanyaan pada decision tree
class Question:

    # inisialisasi kolom dan nilai pertanyaan
    def __init__(self, column, value):
        self.column = column
        self.value = value

    # mengecek apakah contoh data sesuai dengan pertanyaan
    def match(self, example):
        val = example[self.column]
        if is_numeric(val):
            return val >= self.value
        else:
            return val == self.value

    # menampilkan pertanyaan dalam format string yang mudah dibaca
    def __repr__(self):
        condition = "=="
        if is_numeric(self.value):
            condition = ">="
        return "Is %s %s %s?" % (
            header[self.column], condition, str(self.value))

# membagi dataset menjadi dua berdasarkan pertanyaan
def partition(rows, question):
    true_rows, false_rows = [], []
    for row in rows:
        if question.match(row):
            true_rows.append(row)
        else:
            false_rows.append(row)
    return true_rows, false_rows

# menghitung jumlah unique value dari suatu kolom
def class_counts(rows):
    counts = {}
    for row in rows:
        label = row[-1]
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    return counts

# menghitung nilai Gini Impurity untuk sebuah dataset
def gini(rows):
    counts = class_counts(rows)
    impurity = 1
    for lbl in counts:
        prob_of_lbl = counts[lbl] / float(len(rows))
        impurity -= prob_of_lbl**2
    return impurity

# menghitung nilai Information Gain dari pemisahan dataset
def info_gain(left, right, current_uncertainty):
    p = float(len(left)) / (len(left) + len(right))
    return current_uncertainty - p * gini(left) - (1 - p) * gini(right)

# mencari pertanyaan terbaik untuk membagi dataset berdasarkan information gain tertinggi
def find_best_split(rows):
    best_gain = 0
    best_question = None
    current_uncertainty = gini(rows)
    n_features = len(rows[0]) - 1

    for col in range(n_features):
        values = set([row[col] for row in rows])
        for val in values:
            question = Question(col, val)

            # splitting the dataset
            true_rows, false_rows = partition(rows, question)

            # Skip this split if it doesn't divide the dataset
            if len(true_rows) == 0 or len(false_rows) == 0:
                continue
            gain = info_gain(true_rows, false_rows, current_uncertainty)
            if gain >= best_gain:
              best_gain, best_question = gain, question
    return best_gain, best_question

# contoh penggunaan
training_data = [
    ['Green', 3, 'Apple'],
    ['Yellow', 3, 'Apple'],
    ['Red', 1, 'Grape'],
    ['Red', 1, 'Grape'],
    ['Yellow', 3, 'Lemon'],]
best_gain, best_question = find_best_split(training_data)
print(best_gain)
print(best_question)

0.37333333333333324
Is diameter >= 3?


# Fungsi Decision Tree

In [ ]:
# merepresentasikan node daun (leaf) pada decision tree yang berisi hasil prediksi
class Leaf:

    # inisialisasi leaf dengan menghitung jumlah kemunculan tiap kelas
    def __init__(self, rows):
        self.predictions = class_counts(rows)

In [ ]:
# merepresentasikan node keputusan (decision node) yang berisi pertanyaan dan cabang
class Decision_Node:

    # inisialisasi node dengan pertanyaan, cabang benar, dan cabang salah
    def __init__(self,
                 question,
                 true_branch,
                 false_branch):
        self.question = question
        self.true_branch = true_branch
        self.false_branch = false_branch


In [39]:
# label kolom
header = ["color", "diameter", "label"]

# fungsi pengecekan suatu value numerik atau bukan
def is_numeric(value):
    return isinstance(value, int) or isinstance(value, float)

# kelas untuk merepresentasikan pertanyaan pada decision tree
class Question:

    # inisialisasi kolom dan nilai pertanyaan
    def __init__(self, column, value):
        self.column = column
        self.value = value

    # mengecek apakah contoh data sesuai dengan pertanyaan
    def match(self, example):
        val = example[self.column]
        if is_numeric(val):
            return val >= self.value
        else:
            return val == self.value

    # menampilkan pertanyaan dalam format string yang mudah dibaca
    def __repr__(self):
        condition = "=="
        if is_numeric(self.value):
            condition = ">="
        return "Is %s %s %s?" % (
            header[self.column], condition, str(self.value))

# menghitung jumlah unique value dari suatu kolom
def class_counts(rows):
    counts = {}
    for row in rows:
        label = row[-1]
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    return counts

# merepresentasikan node daun (leaf) pada decision tree yang berisi hasil prediksi
class Leaf:

    # inisialisasi leaf dengan menghitung jumlah kemunculan tiap kelas
    def __init__(self, rows):
        self.predictions = class_counts(rows)

# merepresentasikan node keputusan (decision node) yang berisi pertanyaan dan cabang
class Decision_Node:

    # inisialisasi node dengan pertanyaan, cabang benar, dan cabang salah
    def __init__(self,
                 question,
                 true_branch,
                 false_branch):
        self.question = question
        self.true_branch = true_branch
        self.false_branch = false_branch

# mencetak struktur decision tree secara rekursif dalam format teks
def print_tree(node, spacing=""):

    # base case: jika sudah mencapai leaf
    if isinstance(node, Leaf):
        print (spacing + "Predict", node.predictions)
        return

    # mencetak pertanyaan pada node saat ini
    print (spacing + str(node.question))

    # mencetak cabang true secara rekursif
    print (spacing + '--> True:')
    print_tree(node.true_branch, spacing + "  ")

    # mencetak cabang false secara rekursif
    print (spacing + '--> False:')
    print_tree(node.false_branch, spacing + "  ")

# contoh penggunaan
training_data = [
    ['Green', 3, 'Apple'],
    ['Yellow', 3, 'Apple'],
    ['Red', 1, 'Grape'],
    ['Red', 1, 'Grape'],
    ['Yellow', 3, 'Lemon'],]

my_tree = Decision_Node(Question(1, 3), true_branch=Leaf([["Yellow", 3, "Apple"], ["Yellow", 3, "Lemon"]]),
                        false_branch=Leaf([["Red", 1, "Grape"], ["Red", 1, "Grape"]]))
print_tree(my_tree)

Is diameter >= 3?
--> True:
  Predict {'Apple': 1, 'Lemon': 1}
--> False:
  Predict {'Grape': 2}


In [40]:
# mengklasifikasikan satu baris data menggunakan decision tree
def classify(row, node):

    # base case: jika sudah mencapai leaf
    if isinstance(node, Leaf):
      return node

    # menentukan apakah mengikuti cabang true atau cabang false
    # dengan membandingkan nilai fitur pada baris dengan pertanyaan di node
    question = node.question
    if question.match(row):
      return classify(row, node.true_branch)
    else:
      return classify(row, node.false_branch)

# contoh penggunaan
training_data = [
    ['Green', 3, 'Apple'],
    ['Yellow', 3, 'Apple'],
    ['Red', 1, 'Grape'],
    ['Red', 1, 'Grape'],
    ['Yellow', 3, 'Lemon'],]
my_tree = Decision_Node(Question(1, 3), true_branch=Leaf([["Yellow", 3, "Apple"]]),
                        false_branch=Leaf([["Red", 1, "Grape"], ["Yellow", 1, "Lemon"]]))
row1 = training_data[0]
row2 = training_data[1]
print(classify(row1, my_tree).predictions)
print(classify(row2, my_tree).predictions)

{'Apple': 1}
{'Apple': 1}


In [ ]:
# menguji decision tree dengan data uji dan membandingkan hasil prediksi dengan label asli
for row in testing_data:
    prediction = classify(row, my_tree)
    print(f"Actual: {row[-1]}. Predicted: {print_leaf(prediction.predictions)}")

In [3]:
# menampilkan prediksi pada leaf dalam format persentase
def print_leaf(counts):
    total = sum(counts.values()) * 1.0
    probs = {}
    for lbl in counts.keys():
        probs[lbl] = str(int(counts[lbl] / total * 100)) + "%"
        percent = counts[lbl] / total * 100 # define percent here
        if abs(percent - round(percent)) < 1e-9: # add colon here
            probs[lbl] = f"{int(round(percent))}%"
        else:
            probs[lbl] = f"{percent:.1f}%"
    return probs

# contoh penggunaan
# fungsi Menghitung jumlah unique value dari suatu kolom
def class_counts(rows):
    counts = {}
    for row in rows:
        label = row[-1]
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    return counts

training_data = [
    ['Green', 3, 'Apple'],
    ['Yellow', 3, 'Apple'],
    ['Red', 1, 'Grape'],
    ['Red', 1, 'Grape'],
    ['Yellow', 3, 'Lemon'],]
counts = class_counts(training_data)
if_leaf = print_leaf(counts)
print(if_leaf)

{'Apple': '40%', 'Grape': '40%', 'Lemon': '20%'}


# Predict Using Decision Tree

In [37]:
# label kolom
header = ["color", "diameter", "label"]

# fungsi pengecekan suatu value numerik atau bukan
def is_numeric(value):
    return isinstance(value, int) or isinstance(value, float)

# kelas untuk merepresentasikan pertanyaan pada decision tree
class Question:

    # inisialisasi kolom dan nilai pertanyaan
    def __init__(self, column, value):
        self.column = column
        self.value = value

    # mengecek apakah contoh data sesuai dengan pertanyaan
    def match(self, example):
        val = example[self.column]
        if is_numeric(val):
            return val >= self.value
        else:
            return val == self.value

    # menampilkan pertanyaan dalam format string yang mudah dibaca
    def __repr__(self):
        condition = "=="
        if is_numeric(self.value):
            condition = ">="
        return "Is %s %s %s?" % (
            header[self.column], condition, str(self.value))

# menghitung jumlah unique value dari suatu kolom
def class_counts(rows):
    counts = {}
    for row in rows:
        label = row[-1]
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    return counts

# merepresentasikan node daun (leaf) pada decision tree yang berisi hasil prediksi
class Leaf:

    # inisialisasi leaf dengan menghitung jumlah kemunculan tiap kelas
    def __init__(self, rows):
        self.predictions = class_counts(rows)

# merepresentasikan node keputusan (decision node) yang berisi pertanyaan dan cabang
class Decision_Node:

    # inisialisasi node dengan pertanyaan, cabang benar, dan cabang salah
    def __init__(self,
                 question,
                 true_branch,
                 false_branch):
        self.question = question
        self.true_branch = true_branch
        self.false_branch = false_branch

# mengklasifikasikan satu baris data menggunakan decision tree
def classify(row, node):

    # base case: jika sudah mencapai leaf
    if isinstance(node, Leaf):
      return node

    # menentukan apakah mengikuti cabang true atau cabang false
    # dengan membandingkan nilai fitur pada baris dengan pertanyaan di node
    question = node.question
    if question.match(row):
      return classify(row, node.true_branch)
    else:
      return classify(row, node.false_branch)

# menampilkan prediksi pada leaf dalam format persentase
def print_leaf(counts):
    total = sum(counts.values()) * 1.0
    probs = {}
    for lbl in counts.keys():
        probs[lbl] = str(int(counts[lbl] / total * 100)) + "%"
        percent = counts[lbl] / total * 100
        if abs(percent - round(percent)) < 1e-9:
            probs[lbl] = f"{int(round(percent))}%"
        else:
            probs[lbl] = f"{percent:.1f}%"
    return probs

# contoh data uji (testing data)
testing_data = [
    ["Green", 3, "Apple"],
    ["Yellow", 3, "Apple"],
    ["Red", 1, "Grape"],
    ["Yellow", 1, "Lemon"],
]

# Assuming my_tree is built using your build_tree function with your training data.
# Since build_tree is not defined in this cell, I will create a simple tree for demonstration.
# You should replace this with the actual tree built from your training data if needed.
# For this example, I'll use the tree structure from your print_tree example.
my_tree = Decision_Node(Question(1, 3), true_branch=Leaf([["Yellow", 3, "Apple"], ["Yellow", 3, "Lemon"]]),
                        false_branch=Leaf([["Red", 1, "Grape"], ["Red", 1, "Grape"]]))


# menguji decision tree dengan data uji
for row in testing_data:
    prediction = classify(row, my_tree)
    print(f"Actual: {row[-1]}. Predicted: {print_leaf(prediction.predictions)}")

Actual: Apple. Predicted: {'Apple': '50%', 'Lemon': '50%'}
Actual: Apple. Predicted: {'Apple': '50%', 'Lemon': '50%'}
Actual: Grape. Predicted: {'Grape': '100%'}
Actual: Lemon. Predicted: {'Grape': '100%'}
